# ETL Pipeline — Alquiler Barcelona

Orquestador modular. La lógica de cada etapa vive en `src/`:

| Módulo | Responsabilidad |
|--------|-----------------|
| `src/extract/incasol.py` | Parseo de los 4 Excel Incasol (barrios y distritos) |
| `src/transform/macrovars.py` | IPC INE, Euribor 12M BDE, IST Idescat |
| `src/transform/features.py` | YoY, lags, precios reales, dummies regulatorias |
| `src/transform/clean.py` | Filtros LOAD, exclusión de series problemáticas, tipado |
| `src/utils/nulls.py` | Validación y reporte de nulos |

**Datos raw:** `data/data_raw/`  
**Outputs finales:** `data/data_cleaned/`  
**Outputs intermedios:** `data/data_interim/`


In [ ]:
import sys
from pathlib import Path

# Añadir raíz del proyecto al path para importar src/
ROOT = Path().resolve().parent  # notebooks se ejecutan desde src/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW     = ROOT / 'data' / 'data_raw'
DATA_INTERIM = ROOT / 'data' / 'data_interim'
DATA_CLEANED = ROOT / 'data' / 'data_cleaned'
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_CLEANED.mkdir(parents=True, exist_ok=True)

from src.extract.incasol import build_barris_panel, build_district_panel
from src.transform.macrovars import load_ipc, load_euribor, load_ist, merge_macrovars
from src.transform.features import (
    add_time_index, add_growth_features, add_lag_features,
    add_real_prices, add_real_yoy_growth, add_regulation_dummies, add_period_labels
)
from src.transform.clean import clean_districts, clean_barrios
from src.utils.nulls import null_report, validate_no_duplicates

print('Imports OK')
print(f'DATA_RAW:     {DATA_RAW}')
print(f'DATA_INTERIM: {DATA_INTERIM}')
print(f'DATA_CLEANED: {DATA_CLEANED}')


## EXTRACT
Lectura de los 4 Excel Incasol y construcción de los paneles brutos.


In [ ]:
barris_raw   = build_barris_panel(raw_path=DATA_RAW)
district_raw = build_district_panel(raw_path=DATA_RAW)

print(f'barris_raw:   {barris_raw.shape}')
print(f'district_raw: {district_raw.shape}')

validate_no_duplicates(barris_raw,   ['neighborhood_code', 'year', 'quarter'], 'barris')
validate_no_duplicates(district_raw, ['district_code',     'year', 'quarter'], 'districts')

# Guardar intermedios
barris_raw.to_parquet(DATA_INTERIM / 'rental_panel_barris.parquet',  index=False)
district_raw.to_parquet(DATA_INTERIM / 'district_panel.parquet', index=False)
print('Intermedios guardados en data/data_interim/')


## TRANSFORM
Enriquecimiento con macrovariables, feature engineering y precios reales.


In [ ]:
# Cargar macrovariables
ipc_q     = load_ipc(DATA_RAW / 'ipc_ine.csv')
euribor_q = load_euribor(DATA_RAW / 'SeriesBIE[21].csv')
ist_df    = load_ist(DATA_RAW / 'ist14058mun.csv')

print(f'IPC:     {ipc_q.shape}  | rango {ipc_q["date"].min().date()} - {ipc_q["date"].max().date()}')
print(f'Euribor: {euribor_q.shape} | rango {euribor_q["date"].min().date()} - {euribor_q["date"].max().date()}')
print(f'IST:     {ist_df.shape}')

def transform_panel(df_raw, group_col, include_ist=False):
    df = add_time_index(df_raw)
    df = add_growth_features(df, group_col)
    df = add_lag_features(df, group_col)
    df = add_regulation_dummies(df)
    df = merge_macrovars(df, ipc_q, euribor_q, ist_df if include_ist else None)
    df = add_real_prices(df)
    df = add_real_yoy_growth(df, group_col)
    return df

districts_full   = transform_panel(district_raw, 'district_code',     include_ist=False)
barrios_full     = transform_panel(barris_raw,   'neighborhood_code', include_ist=True)

print(f'districts_full: {districts_full.shape}')
print(f'barrios_full:   {barrios_full.shape}')


## LOAD
Filtros finales, etiquetas temporales y guardado en `data/data_cleaned/`.


In [ ]:
df_distritos_eda = add_period_labels(clean_districts(districts_full))
df_barrios_eda   = add_period_labels(clean_barrios(barrios_full))

print(f'df_distritos_eda: {df_distritos_eda.shape}')
print(f'df_barrios_eda:   {df_barrios_eda.shape}')

null_report(df_distritos_eda, 'distritos')
null_report(df_barrios_eda,   'barrios')

df_distritos_eda.to_parquet(DATA_CLEANED / 'df_distritos_eda.parquet', index=False)
df_barrios_eda.to_parquet(  DATA_CLEANED / 'df_barrios_eda.parquet',   index=False)
print('\nOutputs guardados en data/data_cleaned/')
